In [32]:
import yaml
import glob
import json
import os
from dataclasses import dataclass
from pydantic import BaseModel, Field
from pprint import pprint
from typing import Dict, Optional, List, Any, Tuple
from slideguard.schemes import FullEvaluation
from langchain.prompts import ChatPromptTemplate
from langchain_openai import ChatOpenAI
from langchain_core.output_parsers import StrOutputParser, PydanticOutputParser
from langchain_core.language_models import LanguageModelInput

# Functions

In [2]:
class VLLMChatOpenAI(ChatOpenAI):
    def _get_request_payload(
        self,
        input_: LanguageModelInput,
        *,
        stop: Optional[List[str]] = None,
        **kwargs: Any,
    ) -> dict:
        payload = super()._get_request_payload(input_, stop=stop, **kwargs)
        # max_tokens was deprecated in favor of max_completion_tokens
        # in September 2024 release
        if "max_completion_tokens" in payload:
            payload["max_tokens"] = payload.pop("max_completion_tokens")
        return payload
    

def load_goldens(base_path: str = "../golden") -> Dict[str, dict]:
    golden_files = glob.glob(os.path.join(base_path, "*.yaml"))

    goldens = dict()
    for file in golden_files:
        deck_name, _ = os.path.splitext(os.path.basename(file))
        with open(file, "r") as f:
            evaluation = yaml.safe_load(f)
        goldens[deck_name] = evaluation
    
    return goldens


def load_evaluations(base_path: str = "../slidedecks_test_evaluations") -> Dict[str, FullEvaluation]:
    evaluation_files = glob.glob(os.path.join(base_path, "evaluations_*.json"))

    evaluations = dict()
    for file in evaluation_files:
        deck_name, _ = os.path.splitext(os.path.basename(file))
        deck_name = deck_name.replace("evaluations_", "")
        with open(file, "r") as f:
            evaluation = FullEvaluation.model_validate_json(f.read())
        evaluations[deck_name] = evaluation
    
    return evaluations

In [ ]:
system_prompt = """
You are a helpful assistant.
You need to estimate if a golden comment made by a human expert is presented in the set of evaluation comments made by an AI agent. 
The expert's comment may have a different structure and form than the evaluation comment, 
but the essence of the comment should be the same.
AI agent may have many comments in the set, so you need to find at least one comment that is the most similar to the expert's comment.


Write your answer in the following JSON format:
{json_schema}
"""

human_prompt = """
The comment made by a human expert:
{human_comment}

The set of evaluations comment made by an AI agent:
{ai_comments}
"""

class JudgementAnswer(BaseModel):
    citation: List[str] = Field(description="The closest comment (one or more) from the set of evaluation comments to the expert's comment")
    judgement: bool = Field(description="Whether the expert's comment is present in the set of evaluation comments")


llm = VLLMChatOpenAI(
    model="/model",
    temperature=0.1,
    max_completion_tokens=1000,
    max_tokens=1000,
    base_url="http://d.dgx:8082/v1",
    api_key="token-abc123"
)

parser = PydanticOutputParser(pydantic_object=JudgementAnswer)

chat_prompt = ChatPromptTemplate.from_messages([
    ("system", system_prompt),
    ("human", human_prompt)
]).bind(
    json_schema=parser.get_format_instructions()
)

chain = chat_prompt | llm | parser

In [35]:
goldens = load_goldens()
evaluations = load_evaluations()

print("Goldens:", len(goldens))
print("Evaluations:", len(evaluations))

Goldens: 3
Evaluations: 3


In [39]:
from typing import Dict, Optional
from slideguard.schemes import FullEvaluation

def deck_judge_evaluations(goldens: Dict[str, dict], evaluations: Dict[str, FullEvaluation], deck_name: Optional[str] = None):  
    golden2evaluation: Dict[str, Tuple[dict, FullEvaluation]] = dict()
    for deck_name, golden in goldens.items():
        if deck_name in evaluations:
            golden2evaluation[deck_name] = (golden, evaluations[deck_name])
        else:
            print(f"No evaluation for {deck_name}")

    if deck_name is not None:
        if deck_name not in golden2evaluation:
            raise ValueError(f"No data for {deck_name}")
        decks = [deck_name]
    else:
        decks = list(golden2evaluation.keys())

    for deck_name in decks:
        golden, evaluation = golden2evaluation[deck_name]
        for criteria, evaluation in evaluation.deck_evaluations.evaluations.items():
            eval_results = [el for el in evaluation['evaluation_results'] if el['severity'] > 2]
        
            print(f"Evaluating {criteria} for {deck_name}")
            if criteria not in golden['evaluation']['deck']:
                print(f"No golden comment for {criteria} for {deck_name}")
                continue
            
            for comment in golden['evaluation']['deck'][criteria]:
                result = chain.invoke({
                    "human_comment": comment,
                    "ai_comments": str(eval_results)
                })
                print("-" * 100)
                print("Expert comment:", comment)
                print("Result:", result)


In [40]:
deck_judge_evaluations(goldens, evaluations)

Evaluating Criteria.deck_storytelling for 12_EN_Zamiralov_NIR
----------------------------------------------------------------------------------------------------
Expert comment: Нет полноценного интро в решаемую проблему
Result: The closest comment from the set of evaluation comments:
"The presentation lacks a clear connection between the motivation and the goal. The initial slide introduces the topic but does not explicitly state the goal or how it addresses the motivation."

yes
----------------------------------------------------------------------------------------------------
Expert comment: Непонятно, приведенные на слайде 4 и 5 схемы описывают предложенный студентом метод или существующий
Result: No
----------------------------------------------------------------------------------------------------
Expert comment: Обзор литературы не описывает как предлагаемое студентом решение отличается от уже существующих. Обзор должен сравнивать между собой решения.
Result: The closest comme